
Notes:
* Images are DICOM format.
* In TRE, they are contained in zip files so we need to unzip to access images Find the code in the allreadwrite/read_dicom.py file which will do this.
* Files are in parquet format (more efficient data storage and retrieval than csv)
* Use 'df = pd.read_parquet' instead.

-------------------------------------------- Code --------------------------------------------

In [ ]:
# may be installed already
!pip install torchxrayvision
!pip install torchvision
!pip install -U kagglehub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.0/29.0 MB 37.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.5/160.5 kB 10.8 MB/s eta 0:00:00
  Attempting uninstall: kagglehub
    Found existing installation: kagglehub 0.3.13
    Uninstalling kagglehub-0.3.13:
      Successfully uninstalled kagglehub-0.3.13


In [ ]:
import urllib.request

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import skimage
import torch
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
# might not even have torchxrayvision in the TRE environment. Install it using Docker, put in Docker container.
import torchxrayvision as xrv
import kagglehub
import os
import glob
from pathlib import Path

device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
device

device(type='cpu')

In [ ]:
# TRE: loading the dataset at the end will be different because we're using uclh/tre data. this will need to be changed.
# download dataset
dataset = kagglehub.dataset_download("raddar/chest-xrays-indiana-university")

Using Colab cache for faster access to the 'chest-xrays-indiana-university' dataset.


In [ ]:
PATH = Path(dataset)
IMAGE_DIR = PATH / "images" / "images_normalized"
IMAGE_PATHS = glob.glob(os.path.join(IMAGE_DIR, "*.png"))
# print(f"Number of images: ", {len(IMAGE_PATHS)})
# print(f"Example: ", {IMAGE_PATHS[0]})

dataset = torch.utils.data.Dataset(IMAGE_PATHS)
# create data loaders
# pin_memory=True is used to speed up the transfer of data to the GPU in cuda
data_loader = torch.utils.data.DataLoader(dataset, batch_size=64, shuffle=True)

Number of images:  {7470}
Example:  {'/kaggle/input/chest-xrays-indiana-university/images/images_normalized/349_IM-1697-2001.dcm.png'}


In [ ]:
# ===== CONFIGURATION (Top of notebook) =====
# import torch
# import torchxrayvision as xrv
# from PIL import Image
# import numpy as np

# Device configuration
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(DEVICE)

# Model configuration
MODEL_WEIGHTS = "densenet121-res224-all"

# Difficulty thresholds
EASY_THRESHOLD = 0.85
HARD_THRESHOLD = 0.60

# ===== MODEL LOADING =====
model_cnn = xrv.models.DenseNet(weights=MODEL_WEIGHTS)
model_cnn = model_cnn.to(DEVICE)
model_cnn.eval()

# Pathologies (matches model output)
try:
    PATHOLOGIES = model_cnn.pathologies
    print("Retrieved pathology labels.")
except Exception as e:
    print("Could not retrieve labels. Using fallback labels.")
    PATHOLOGIES = [
         "Atelectasis", "Cardiomegaly", "Consolidation", "Edema",
         "Emphysema", "Fibrosis", "Hernia", "Infiltration", "Mass",
         "Nodule", "Pleural Effusion", "Pneumonia", "Pneumothorax", "No Finding"
    ]

# ===== FUNCTIONS =====
# def assign_difficulty(confidence):
#     """Uses global thresholds"""
#     if confidence > EASY_THRESHOLD:
#         return 'EASY'
#     elif confidence < HARD_THRESHOLD:
#         return 'HARD'
#     else:
#         return 'MODERATE'

def load_xray(img_path):
    """
    Load and preprocess a chest X-ray image for model inference.

    Applies standard preprocessing pipeline: load → normalize → grayscale →
    add channel → center crop → resize (224×224) → convert to tensor.

    Args:
        img_path: Path to X-ray image file (.jpg, .png, .dcm)

    Returns:
        torch.Tensor: Preprocessed image (1, 224, 224), float32, normalized

    Example:
        >>> img = load_xray("data/patient_01/images/xray_001.jpg")
        >>> predictions = model(img.unsqueeze(0))
    """
    # Load and normalize
    img = skimage.io.imread(img_path)
    img = xrv.datasets.normalize(img, 255)

    # Convert RGB to grayscale if needed
    if img.ndim == 3 and img.shape[2] == 3:
        img = img.mean(axis=2)

    # Add channel dimension
    if img.ndim == 2:
        img = img[None, ...]

    # Crop and resize
    transform = transforms.Compose([
        xrv.datasets.XRayCenterCrop(),
        xrv.datasets.XRayResizer(224)
    ])
    img = transform(img)

    # Convert to tensor
    return torch.from_numpy(img.astype(np.float32))

def predict_xray(img_tensor, model, device):
    """Uses global PATHOLOGIES"""
    try:
        with torch.no_grad():
            img_tensor = img_tensor.to(device)
            preds = model(img_tensor.unsqueeze(0))
            pathology_scores = preds[0].cpu().numpy()

            results = {
                label: float(score)
                for label, score in zip(PATHOLOGIES, pathology_scores)
            }

            top_label = max(results, key=results.get)
            judgement = f"Top prediction: {top_label}"

            return judgement, results
    except Exception as e:
        return f"Error: {str(e)}", None

def process_images_with_difficulty(
    image_paths,
    model,
    device,
    easy_threshold=0.85,
    hard_threshold=0.60,
    limit=None,
    show_progress=True
):
  results = []

  # Apply limit if specified
  paths_to_process = image_paths[:limit] if limit else image_paths
  total = len(paths_to_process)

  print(f"Processing {total} images...")

  for img_path in paths_to_process:
      try:
          # Load and preprocess
          img_tensor = load_xray(img_path)

          # Get predictions
          judgement, scores = predict_xray(img_tensor, model, device)

          # Extract confidence and top prediction
          max_confidence = max(scores.values())
          top_prediction = max(scores, key=scores.get)

          if max_confidence > easy_threshold:
                difficulty = 'EASY'
          elif max_confidence < hard_threshold:
              difficulty = 'HARD'
          else:
              difficulty = 'MODERATE'

          # Store results
          results.append({
              'image_path': img_path,
              'max_confidence': max_confidence,
              'top_prediction': top_prediction,
              'difficulty': difficulty,
              'all_scores': scores
          })

      except Exception as e:
          print(f"Error processing {img_path}: {str(e)}")

  df = pd.DataFrame(results)

  return df

cpu
If this fails you can run `wget https://github.com/mlmed/torchxrayvision/releases/download/v1/nih-pc-chex-mimic_ch-google-openi-kaggle-densenet121-d121-tw-lr001-rot45-tr15-sc15-seed0-best.pt -O /root/.torchxrayvision/models_data/nih-pc-chex-mimic_ch-google-openi-kaggle-densenet121-d121-tw-lr001-rot45-tr15-sc15-seed0-best.pt`
[██████████████████████████████████████████████████]
Retrieved pathology labels.


In [ ]:
# ===== USAGE =====
# Easy to call - no need to pass pathologies
img_tensor = load_xray(IMAGE_PATHS[0])
df_cv_results = process_images_with_difficulty(img_tensor, model_cnn, DEVICE)

# print(f"Confidence: {confidence:.2f}")
# print(f"Difficulty: {difficulty}")

# process_images_with_difficulty(IMAGE_PATHS, model_cnn, DEVICE, limit=5)

Processing 1 images...
Error processing tensor([[ 1.1547e+01,  2.6948e+01,  3.9514e+01,  ..., -8.1704e+02,
         -8.1328e+02, -7.0721e+02],
        [ 6.7749e+00,  2.1636e+01,  3.9228e+01,  ..., -9.4144e+02,
         -9.3552e+02, -8.1316e+02],
        [-8.0593e-01,  1.3193e+01,  2.9147e+01,  ..., -9.4351e+02,
         -9.3673e+02, -8.1346e+02],
        ...,
        [ 6.3743e+02,  7.3033e+02,  7.2605e+02,  ..., -9.0261e+02,
         -8.9642e+02, -7.7854e+02],
        [ 5.4011e+02,  6.2350e+02,  6.2013e+02,  ..., -9.0189e+02,
         -8.9570e+02, -7.7826e+02],
        [ 4.4314e+02,  5.1517e+02,  5.2643e+02,  ..., -7.8458e+02,
         -7.7921e+02, -6.7686e+02]]): Cannot understand given URI: tensor([[ 1.1547e+01,  2.6948e+01,  3.9514e+01,  ..., -8.....
Processing 5 images...


,image_path,max_confidence,top_prediction,difficulty,all_scores
0,/kaggle/input/chest-xrays-indiana-university/i...,0.853788,Lung Opacity,EASY,"{'Atelectasis': 0.5466604232788086, 'Consolida..."
1,/kaggle/input/chest-xrays-indiana-university/i...,0.509079,Fibrosis,HARD,"{'Atelectasis': 0.5078667402267456, 'Consolida..."
2,/kaggle/input/chest-xrays-indiana-university/i...,0.629605,Lung Opacity,MODERATE,"{'Atelectasis': 0.531897783279419, 'Consolidat..."
3,/kaggle/input/chest-xrays-indiana-university/i...,0.737883,Lung Opacity,MODERATE,"{'Atelectasis': 0.5151936411857605, 'Consolida..."
4,/kaggle/input/chest-xrays-indiana-university/i...,0.656385,Lung Opacity,MODERATE,"{'Atelectasis': 0.5761685371398926, 'Consolida..."
